# Tune Hybrid Weights + Segment Evaluation
Grid search weight cho collab/content/pop/artist để tối ưu F1@10, đồng thời đánh giá theo segment user/item.

In [1]:
# %pip install pandas numpy sqlalchemy pymysql python-dotenv

In [2]:
import os
from collections import defaultdict
from itertools import product
from urllib.parse import parse_qsl, quote_plus, urlencode, urlparse
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

In [3]:
def mk_url(db_url, user, pwd):
    if db_url.startswith('jdbc:'): db_url = db_url[5:]
    p = urlparse(db_url)
    q = dict(parse_qsl(p.query, keep_blank_values=True))
    m = {}
    if 'useSSL' in q:
        m['ssl_disabled'] = str((q['useSSL'].lower() != 'true')).lower()
    m.setdefault('charset','utf8mb4')
    m.setdefault('connect_timeout','10')
    return f"mysql+pymysql://{quote_plus(user)}:{quote_plus(pwd)}@{p.hostname}:{p.port or 3306}/{p.path.lstrip('/')}?{urlencode(m)}"

load_dotenv(find_dotenv(usecwd=True), override=False)
engine = create_engine(mk_url(os.getenv('DB_URL'), os.getenv('DB_USERNAME'), os.getenv('DB_PASSWORD')), pool_pre_ping=True)
print('connected')

connected


In [4]:
ACTION_WEIGHT = {0:1.0,1:3.0,2:4.0,3:2.0,4:5.0}
premium_song_type = int(os.getenv('PREMIUM_SONG_TYPE','1'))
premium_boost_multiplier = float(os.getenv('PREMIUM_BOOST_MULTIPLIER','1.35'))
premium_min_slots = int(os.getenv('PREMIUM_MIN_SLOTS','2'))

In [5]:
inter = pd.read_sql(text('SELECT user_id, song_id, action_type, created_at FROM interaction_song WHERE user_id IS NOT NULL'), engine)
inter_playlist = pd.read_sql(text('SELECT user_id, playlist_id, action_type, COALESCE(interaction_count,1) AS interaction_count FROM interaction_playlist WHERE user_id IS NOT NULL'), engine)
playlist_song = pd.read_sql(text('SELECT playlist_id, song_id FROM playlist_song'), engine)
inter_artist = pd.read_sql(text('SELECT user_id, artist_id, COALESCE(interaction_count,0) AS interaction_count FROM interaction_artist WHERE user_id IS NOT NULL'), engine)
song = pd.read_sql(text('SELECT id AS song_id, type, language, deleted_at FROM song'), engine)
genre_song = pd.read_sql(text('SELECT song_id, genre_id FROM genre_song'), engine)
artist_song = pd.read_sql(text('SELECT song_id, artist_id FROM artist_song'), engine)
audio_q = pd.read_sql(text('SELECT song_id, SUM(CASE WHEN type=1 THEN 1 ELSE 0 END) AS high_q_cnt, SUM(CASE WHEN url IS NOT NULL AND url <> "" THEN 1 ELSE 0 END) AS playable_cnt FROM audio_quality GROUP BY song_id'), engine)
inter['created_at'] = pd.to_datetime(inter['created_at'], errors='coerce')
inter = inter.dropna(subset=['created_at','user_id','song_id'])

In [6]:
# normalize language
song['language_norm'] = song['language'].fillna('unknown').astype(str).str.strip().str.lower().replace({'': 'unknown', 'none': 'unknown', 'null': 'unknown'})
active_song = set(song[song['deleted_at'].isna()]['song_id'])
premium_set = set(song[song['type'] == premium_song_type]['song_id'])
audio_map = audio_q.set_index('song_id')[['high_q_cnt','playable_cnt']].to_dict('index')

In [7]:
cut = inter['created_at'].quantile(0.8)
train = inter[inter['created_at'] <= cut].copy()
test = inter[inter['created_at'] > cut].copy()
valid_users = set(train['user_id'].unique()) & set(test['user_id'].unique())
train = train[train['user_id'].isin(valid_users)]
test = test[test['user_id'].isin(valid_users)]
print(len(train), len(test), len(valid_users))

11784 7685 301


In [8]:
train['w'] = train['action_type'].map(ACTION_WEIGHT).fillna(1.0)
train_seen = train.groupby('user_id')['song_id'].apply(set).to_dict()
truth = test.groupby('user_id')['song_id'].apply(set).to_dict()
pop = train.groupby('song_id')['w'].sum().to_dict()
user_hist = train.groupby('user_id')['song_id'].apply(list).to_dict()
song_users = defaultdict(set)
for r in train[['user_id','song_id']].itertuples(index=False):
    song_users[r.song_id].add(r.user_id)
song_genres = genre_song.groupby('song_id')['genre_id'].apply(set).to_dict()
song_artists = artist_song.groupby('song_id')['artist_id'].apply(set).to_dict()
artist_pref = inter_artist.groupby(['user_id','artist_id'])['interaction_count'].sum().reset_index()
artist_pref_map = defaultdict(dict)
for r in artist_pref.itertuples(index=False):
    artist_pref_map[r.user_id][r.artist_id] = float(r.interaction_count)

# playlist signal: user -> songs from interacted playlists
pl_signal = inter_playlist.merge(playlist_song, on='playlist_id', how='left')
pl_score = pl_signal.groupby(['user_id','song_id'])['interaction_count'].sum().reset_index()
playlist_pref_map = defaultdict(dict)
for r in pl_score.itertuples(index=False):
    playlist_pref_map[r.user_id][r.song_id] = float(r.interaction_count)

In [9]:
all_song_ids = set(song['song_id'].unique())
user_event_count = train.groupby('user_id').size().to_dict()
q80 = np.quantile(list(user_event_count.values()), 0.8)
q20 = np.quantile(list(user_event_count.values()), 0.2)

def user_segment(u):
    c = user_event_count.get(u, 0)
    if c <= q20: return 'cold'
    if c >= q80: return 'heavy'
    return 'warm'

item_pop = train.groupby('song_id').size().to_dict()
iq80 = np.quantile(list(item_pop.values()), 0.8)

def item_segment(sid):
    return 'popular' if item_pop.get(sid,0) >= iq80 else 'long_tail'

In [10]:
def apply_audio(scores):
    out = {}
    for sid, sc in scores.items():
        if sid not in active_song: continue
        aq = audio_map.get(sid, {'high_q_cnt':0,'playable_cnt':0})
        if float(aq['playable_cnt']) <= 0: continue
        qb = 1.0 + min(float(aq['playable_cnt']),3.0)*0.05 + min(float(aq['high_q_cnt']),2.0)*0.08
        out[sid] = float(sc) * qb
    return out

def apply_premium(scores):
    return {sid: (float(sc)*premium_boost_multiplier if sid in premium_set else float(sc)) for sid,sc in scores.items()}

def apply_pop_bias(u, scores):
    seg = user_segment(u)
    if seg == 'heavy':
        # giảm bias popular cho heavy users
        return {sid: sc * (0.85 if item_segment(sid)=='popular' else 1.08) for sid,sc in scores.items()}
    if seg == 'cold':
        # tăng nhẹ bài popular cho cold users
        return {sid: sc * (1.08 if item_segment(sid)=='popular' else 0.98) for sid,sc in scores.items()}
    return scores

def topk(scores, k):
    ranked = [sid for sid,_ in sorted(scores.items(), key=lambda t:t[1], reverse=True)]
    top = ranked[:k]
    prem = [s for s in top if s in premium_set]
    need = max(0, premium_min_slots - len(prem))
    if need > 0:
        backup = [s for s in ranked[k:] if s in premium_set][:need]
        non = [s for s in top if s not in premium_set]
        top = (prem + backup + non)[:k]
    return top

In [11]:
def scorer_parts(u, exclude):
    hist = user_hist.get(u, [])[:30]

    collab = defaultdict(float)
    for sid in hist:
        for ou in song_users.get(sid, set()):
            if ou == u: continue
            for csid in train_seen.get(ou, set()):
                if csid != sid and csid not in exclude:
                    collab[csid] += 1.0

    pg, pa = set(), set()
    for sid in hist:
        pg |= song_genres.get(sid, set())
        pa |= song_artists.get(sid, set())
    content = {}
    for sid in all_song_ids:
        if sid in exclude: continue
        go = len(pg & song_genres.get(sid, set()))
        ao = len(pa & song_artists.get(sid, set()))
        if go or ao: content[sid] = ao*2 + go

    pop_part = {sid: v for sid, v in pop.items() if sid not in exclude}

    artist_part = defaultdict(float)
    ap = artist_pref_map.get(u, {})
    for sid, artists in song_artists.items():
        if sid in exclude: continue
        for a in artists:
            artist_part[sid] += ap.get(a, 0.0)

    playlist_part = playlist_pref_map.get(u, {})

    return collab, content, pop_part, artist_part, playlist_part

In [12]:
def merge_parts(parts, w):
    c,t,p,a,pl = parts
    wc,wt,wp,wa,wpl = w
    s = defaultdict(float)
    for k,v in c.items(): s[k] += wc*v
    for k,v in t.items(): s[k] += wt*v
    for k,v in p.items(): s[k] += wp*v
    for k,v in a.items(): s[k] += wa*v
    for k,v in pl.items(): s[k] += wpl*v
    return s

def metrics_at_k(rec, gt, k=10):
    r = rec[:k]
    if not r: return 0.0,0.0,0.0
    h = len(set(r) & set(gt))
    p = h/k
    rr = h/len(gt) if gt else 0.0
    f = (2*p*rr/(p+rr)) if (p+rr)>0 else 0.0
    return p,rr,f

In [15]:
# Grid search: collab/content/pop/artist/playlist
grid = []
vals = [0.1,0.2,0.3,0.4,0.5,0.6]
for wc,wt,wp,wa,wpl in product(vals, vals, vals, vals, vals):
    s = wc+wt+wp+wa+wpl
    if abs(s-1.0) < 1e-9:
        grid.append((wc,wt,wp,wa,wpl))

users = list(truth.keys())[:50]
cache = {u: scorer_parts(u, train_seen.get(u,set())) for u in users}

rows = []
for w in grid:
    fvals = []
    for u in users:
        sc = merge_parts(cache[u], w)
        sc = apply_audio(sc)
        sc = apply_premium(sc)
        sc = apply_pop_bias(u, sc)
        rec = topk(sc, 10)
        _,_,f = metrics_at_k(rec, truth.get(u,set()), 10)
        fvals.append(f)
    rows.append({'w':w, 'F1@10': float(np.mean(fvals))})

grid_res = pd.DataFrame(rows).sort_values('F1@10', ascending=False).reset_index(drop=True)
grid_res.head(15)

,w,F1@10
0,"(0.5, 0.1, 0.1, 0.2, 0.1)",0.121369
1,"(0.5, 0.1, 0.1, 0.1, 0.2)",0.121369
2,"(0.5, 0.2, 0.1, 0.1, 0.1)",0.121369
3,"(0.4, 0.1, 0.1, 0.1, 0.3)",0.121015
4,"(0.1, 0.2, 0.1, 0.2, 0.4)",0.119968
5,"(0.3, 0.1, 0.1, 0.2, 0.3)",0.119968
6,"(0.3, 0.2, 0.1, 0.1, 0.3)",0.119968
7,"(0.1, 0.2, 0.1, 0.3, 0.3)",0.119968
8,"(0.3, 0.1, 0.1, 0.1, 0.4)",0.119968
9,"(0.2, 0.1, 0.1, 0.2, 0.4)",0.119968


In [16]:
best_w = grid_res.iloc[0]['w']
best_w

(0.5, 0.1, 0.1, 0.2, 0.1)

In [18]:
def eval_segments(weights, k=10, max_users=300):
    users = list(truth.keys())[:max_users]
    out = []
    for seg in ['cold', 'warm', 'heavy']:
        pu, ru, fu = [], [], []
        for u in users:
            if user_segment(u) != seg:
                continue
            parts = cache.get(u)
            if parts is None:
                parts = scorer_parts(u, train_seen.get(u, set()))
                cache[u] = parts
            sc = merge_parts(parts, weights)
            sc = apply_audio(sc)
            sc = apply_premium(sc)
            sc = apply_pop_bias(u, sc)
            rec = topk(sc, k)
            p, r, f = metrics_at_k(rec, truth.get(u, set()), k)
            pu.append(p); ru.append(r); fu.append(f)
        out.append({
            'segment': 'user_' + seg,
            'P@10': np.mean(pu) if pu else 0.0,
            'R@10': np.mean(ru) if ru else 0.0,
            'F1@10': np.mean(fu) if fu else 0.0,
            'users': len(pu)
        })
    for item_seg in ['popular', 'long_tail']:
        pvals, rvals, fvals = [], [], []
        for u in users:
            parts = cache.get(u)
            if parts is None:
                parts = scorer_parts(u, train_seen.get(u, set()))
                cache[u] = parts
            sc = merge_parts(parts, weights)
            sc = apply_audio(sc)
            sc = apply_premium(sc)
            sc = apply_pop_bias(u, sc)
            rec = topk(sc, k)
            rec_seg = [sid for sid in rec if item_segment(sid) == item_seg]
            gt_seg = {sid for sid in truth.get(u, set()) if item_segment(sid) == item_seg}
            kk = max(1, min(k, len(rec_seg)))
            p, r, f = metrics_at_k(rec_seg, gt_seg, kk)
            pvals.append(p); rvals.append(r); fvals.append(f)
        out.append({
            'segment': 'item_' + item_seg,
            'P@10': np.mean(pvals) if pvals else 0.0,
            'R@10': np.mean(rvals) if rvals else 0.0,
            'F1@10': np.mean(fvals) if fvals else 0.0,
            'users': len(users)
        })
    return pd.DataFrame(out)

seg_res = eval_segments(best_w, k=10)
seg_res

,segment,P@10,R@10,F1@10,users
0,user_cold,0.091209,0.242784,0.109239,91
1,user_warm,0.103448,0.190813,0.096016,145
2,user_heavy,0.143750,0.100308,0.069304,64
3,item_popular,0.108333,0.217259,0.107725,300
4,item_long_tail,0.000000,0.000000,0.000000,300
